# 這是把相機拍的照片自動裁切出一顆一顆豆子照片的程式
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arc13Tangent/Pu-Tai-Coffee-Machine-Learning/blob/main/From_Original_to_Crop_Colab.ipynb)

## 步驟 0：把課程的圖片下載下來

這一格會把老師準備好的咖啡豆照片，從 GitHub 下載到這台 Colab 電腦上。

按左邊的 ▶ 執行它（或按 `Shift + Enter`），看到 `✅ 圖片準備完成！` 就成功了。

In [ ]:
# 從 GitHub 下載課程資料（圖片）到 Colab
# 如果你重複執行這一格出現「already exists」，是正常的，代表已經載過了
![ -d Pu-Tai-Coffee-Machine-Learning ] && echo '已經載過了，跳過' || git clone --depth 1 https://github.com/Arc13Tangent/Pu-Tai-Coffee-Machine-Learning.git

print('✅ 圖片準備完成！')

## 環境設定

### 字型設定

In [ ]:
# Colab 預設沒有中文字型，先安裝（第一次跑約 10-20 秒）
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager

# 找系統上有的中文字型
def find_cjk_fonts():
    return [f.name for f in font_manager.fontManager.ttflist
            if any(kw in f.name for kw in ['Noto', 'CJK', 'Heiti', 'PingFang', 'WenQuanYi'])]

fonts = find_cjk_fonts()

# 如果找不到（Colab 通常找不到），就安裝 Noto CJK 再重新偵測
if not fonts:
    print('沒有中文字型，正在安裝...')
    import subprocess
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'fonts-noto-cjk'], check=False)
    # 讓 matplotlib 重新掃描字型
    font_manager.fontManager.__init__()
    fonts = find_cjk_fonts()

print('可用中文字型：', fonts)

# 挑第一個用
if fonts:
    matplotlib.rc('font', family=fonts[0])
    matplotlib.rcParams['axes.unicode_minus'] = False  # 讓負號正常顯示
    print('✅ 中文字型設定完成')
else:
    print('⚠️ 還是找不到中文字型，圖上的中文可能會變方框，但不影響程式執行')


### 套件

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

##  前處理之前的觀察

### 設定看好豆還是壞豆

In [ ]:
# 課程圖片的位置（從 GitHub 下載下來的資料夾）
# 結構：Images/Raw/Normal（好豆）、Images/Raw/Defective（壞豆）
BASE = Path("Pu-Tai-Coffee-Machine-Learning/Images")


In [ ]:
# 要看好豆就留好豆的那一句，要看壞豆就留壞豆的那一句
#TYPE = "G" # Good: Normal 
TYPE = "B" # Bad: Defective

# --- 自動路徑設定，不需要動下面 -----------
if TYPE == "G":
    SRC_DIR = BASE / "Raw" / "Normal"
    img_paths = sorted(SRC_DIR.glob("*.jpg")) + sorted(SRC_DIR.glob("*.png")) + sorted(SRC_DIR.glob("*.JPG"))
    print(f"現在看的是「好豆」in {str(SRC_DIR)}")
    
elif TYPE == "B":
    SRC_DIR = BASE / "Raw" / "Defective"
    img_paths = sorted(SRC_DIR.glob("*.jpg")) + sorted(SRC_DIR.glob("*.png")) + sorted(SRC_DIR.glob("*.JPG"))
    print(f"現在看的是「壞豆」in {str(SRC_DIR)}")
else:
    print("打錯字了！")
    del SRC_DIR, img_paths


In [ ]:
# 就先看第n張
n = 2
img = cv2.imread(str(img_paths[n]))
# OpenCV 存圖片是用 BGR 而不是 RGB！
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(8, 8))
plt.imshow(img_rgb)
plt.title(f"原圖：{img_paths[n].name}  ({img.shape[1]}×{img.shape[0]})")
plt.axis("off")
plt.tight_layout()
plt.show()

### 觀察BGR

In [ ]:
# OpenCV 存圖片是用 BGR 而不是 RGB！
blue   = img[:,:,0]
green  = img[:,:,1]
red    = img[:,:,2]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(blue,  cmap="gray"); axes[0].set_title("B")
axes[1].imshow(green, cmap="gray"); axes[1].set_title("G")
axes[2].imshow(red,   cmap="gray"); axes[2].set_title("R")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

### 決定用___當作裁切用的灰階值
Exercise: ___ 想填什麼？

In [ ]:
gray    = img[:,:,0]
blurred = cv2.GaussianBlur(gray, (5, 5), 0) # 高斯模糊，避免太細節的東西把豆子分成兩半

# thresh, binary_img = cv2.threshold(src,    thresh,  255,                 cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
# 域值  , 黑白圖      =               圖片,    域值,     超過域值的要設成多少 ,  反轉黑白               + 域值由 OTSU 演算法給定

# 原本域值要自己給一個定值，但我們有 OTSU 演算法可以自動算，所以下面的域值欄位填 0 只是佔位用的
# 我們目標是豆子，所以用反轉(cv2.THRESH_BINARY_INV)把「白背景、黑豆子」轉成「黑背景、白豆子」
_, binary = cv2.threshold(blurred, 0, 255, 
                           cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(gray,    cmap="gray"); axes[0].set_title("灰階")
axes[1].imshow(blurred, cmap="gray"); axes[1].set_title("高斯模糊後")
axes[2].imshow(binary,  cmap="gray"); axes[2].set_title("Otsu 二值化")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

### 做mask

In [ ]:
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7)) # 刷子型狀是cv2.MORPH_ELLIPSE(橢圓形)，大小 7*7 pixel

# 閉運算：先膨脹（dilate）再侵蝕（erode）
# 膨脹：把豆子可能的缺塊補起來
# 侵蝕：削掉邊緣
# 為什麼要做這件事：請看對話記錄
closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel) # 形態學(圖片, 閉運算, 使用的 kernel)

# cv2.RETR_EXTERNAL：只找最外層的輪廓
# cv2.CHAIN_APPROX_SIMPLE：只是省記憶體用的
contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# 小於這個面積被視為雜訊忽略
MIN_AREA = 200
valid = [c for c in contours if cv2.contourArea(c) > MIN_AREA]

# 在原圖上畫出偵測到的 bounding box
vis = img_rgb.copy()
for cnt in valid:
    # boundingRect 會找出能框住這個輪廓的最小矩形
    # (x,y): 左上角的點
    # (w, h): 寬、高
    x, y, w, h = cv2.boundingRect(cnt) 
    # cv2.rectangle: 只是畫矩形出來讓我們看一下而已
    cv2.rectangle(vis, (x, y), (x+w, y+h), (255, 80, 80), 2)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(closed, cmap="gray"); axes[0].set_title("閉運算後的 mask")
axes[1].imshow(vis);                 axes[1].set_title(f"偵測到 {len(valid)} 顆豆子")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

print(f"contour 總數（含雜訊）：{len(contours)}")
print(f"MIN_AREA={MIN_AREA} 篩選後：{len(valid)} 顆")

### 切

In [ ]:
PADDING = 0
# img.shape 回傳 (高度, 寬度, 色彩通道數)，[:2] 只取前兩個，所以 h_img 是圖片高度、w_img 是寬度。之後裁切時用來確保不會超出圖片邊界。
h_img, w_img = img.shape[:2]

bboxes = sorted(
    [(cv2.boundingRect(c)) for c in valid],
    key=lambda b: (b[1], b[0])
)

crops = []
for x, y, w, h in bboxes[:20]: # 這邊只看前20顆
    x1, y1 = max(x-PADDING, 0), max(y-PADDING, 0)
    x2, y2 = min(x+w+PADDING, w_img), min(y+h+PADDING, h_img)
    crops.append(img_rgb[y1:y2, x1:x2])

cols = 5
rows = (len(crops) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 3))
axes = axes.flatten()

for i, crop in enumerate(crops):
    axes[i].imshow(crop)
    axes[i].set_title(f"#{i+1}", fontsize=9)
    axes[i].axis("off")

for ax in axes[len(crops):]:
    ax.axis("off")

plt.suptitle("裁切預覽（前 20 顆，依位置排序）")
plt.tight_layout()
plt.show()

## 一切看起來沒問題，正式切

In [ ]:
# ── 路徑設定 ──────────────────────────────────────────────
# 切出來的豆子會存到 Colab 的 Output 資料夾（不會動到原本的圖）
OUTPUT = Path("Output")

if TYPE == "G":
    postfix = "Normal"
    SRC_DIR  = BASE   / "Raw" / postfix
    DST_DIR  = OUTPUT / "Crop_Images" / postfix
    MASK_DIR = OUTPUT / "Crop_Images" / (postfix + "_Mask")
    img_paths = sorted(SRC_DIR.glob("*.jpg")) + sorted(SRC_DIR.glob("*.png")) + sorted(SRC_DIR.glob("*.JPG"))
    print(f"現在切的是「好豆」in {str(SRC_DIR)}")
    
elif TYPE == "B":
    postfix = "Defective"
    SRC_DIR  = BASE   / "Raw" / postfix
    DST_DIR  = OUTPUT / "Crop_Images" / postfix
    MASK_DIR = OUTPUT / "Crop_Images" / (postfix + "_Mask")
    img_paths = sorted(SRC_DIR.glob("*.jpg")) + sorted(SRC_DIR.glob("*.png")) + sorted(SRC_DIR.glob("*.JPG"))
    print(f"現在切的是「壞豆」in {str(SRC_DIR)}")
else:
    print("打錯字了！")
    del postfix, SRC_DIR, DST_DIR, MASK_DIR, img_paths

DST_DIR.mkdir(parents=True, exist_ok=True)
MASK_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# ── 主流程 ───────────────────────────────────────────────
#img_paths = sorted(SRC_DIR.glob("*.jpg")) + sorted(SRC_DIR.glob("*.png")) + sorted(SRC_DIR.glob("*.JPG"))

for img_path in img_paths:
    img_id = img_path.stem
    img    = cv2.imread(str(img_path))
    gray   = img[:,:,0]
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    _, binary = cv2.threshold(blurred, 0, 255,
                              cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    bboxes = []
    for cnt in contours:
        if cv2.contourArea(cnt) < MIN_AREA:
            continue
        x, y, w, h = cv2.boundingRect(cnt)
        bboxes.append((y, x, w, h, cnt))   # ← 把 cnt 一起帶著
    bboxes.sort()

    h_img, w_img = img.shape[:2]
    for bean_idx, (y, x, w, h, cnt) in enumerate(bboxes, start=1):
        x1 = max(x - PADDING, 0)
        y1 = max(y - PADDING, 0)
        x2 = min(x + w + PADDING, w_img)
        y2 = min(y + h + PADDING, h_img)

        # 裁切原圖
        crop = img[y1:y2, x1:x2]
        out_name = f"{img_id}_{bean_idx:04d}.jpg"
        cv2.imwrite(str(DST_DIR / out_name), crop)

        # 對應的 mask：把 cnt 畫在跟 crop 同樣大小的空白圖上
        mask_full = np.zeros(img.shape[:2], dtype=np.uint8)
        cv2.drawContours(mask_full, [cnt], -1, 255, thickness=cv2.FILLED)
        mask_crop = mask_full[y1:y2, x1:x2]
        cv2.imwrite(str(MASK_DIR / out_name), mask_crop)   # 同檔名，不同資料夾

    print(f"[{img_path.name}] → {len(bboxes)} 顆豆子")

print("✅ 完成！")

## 把切好的豆子下載到自己的電腦

執行下面這一格，會把剛剛切出來的所有豆子打包成一個 zip 檔，自動下載到你電腦的「下載」資料夾。

In [ ]:
# 把 Output 資料夾打包成 zip
import shutil

zip_path = shutil.make_archive("coffee_beans_crop", "zip", "Output")
print(f"✅ 打包完成：{zip_path}")

# 下載到自己的電腦（只在 Colab 上有作用）
try:
    from google.colab import files
    files.download(zip_path)
    print("📥 開始下載到你的電腦...")
except ImportError:
    print("（不是在 Colab 上執行，檔案就放在左邊的檔案區）")
